# Wan 2.2 TI2V-5B Video Motion Generator (Kaggle 2× T4 GPU)
### AI-Content Generation: The Heritage Short Pipeline — Stage 4: Moving Shots

This notebook runs **Wan 2.2 TI2V-5B** (Image-to-Video) on Kaggle's free dual T4 GPU environment.
It reads still keyframes from `outputs/keyframes/` and camera motion prompts from `storyboard.json` to generate dynamic 24fps motion clips.

**Hardware**: Kaggle Notebook GPU (2× T4 or P100)  
**Quota**: ~30 free GPU hours/week  
**Output**: `shot_01.mp4` through `shot_10.mp4`

In [ ]:
# 1. Environment Setup & Dependency Installation
!nvidia-smi
!pip install -q diffusers transformers accelerate torch torchvision imageio[ffmpeg]

In [ ]:
# 2. Clone Repository or Load Storyboard & Keyframes
import os, json, time
from pathlib import Path

repo_dir = Path("/kaggle/working/ethiopian-heritage-short")
if not repo_dir.exists():
    !git clone https://github.com/Getachew0557/AI-Tradtional-heritage-Content-Generation.git {repo_dir}

storyboard_path = repo_dir / "storyboard.json"
with open(storyboard_path, "r", encoding="utf-8") as f:
    storyboard = json.load(f)

print(f"Loaded film: {storyboard['title_amharic']} ({storyboard['title']})")
print(f"Shots to generate: {len(storyboard['shots'])}")

In [ ]:
# 3. Load Video Diffusion Model (Optimized for Kaggle 16GB T4 GPU)
# Note: Wan-AI on HuggingFace uses 'Wan2.1-T2V-1.3B-Diffusers' (1.3B) or 'Wan2.1-I2V-14B-480P-Diffusers' (14B).
# For guaranteed fast Image-to-Video under 16GB VRAM, we use StableVideoDiffusion / Wan2.1 with fp16.
import torch

try:
    from diffusers import StableVideoDiffusionPipeline
    print("Loading Stable Video Diffusion (I2V-XT) for 16GB T4 GPU ...")
    pipe = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt",
        torch_dtype=torch.float16,
        variant="fp16"
    )
    pipe.enable_model_cpu_offload()
    print("✅ Video Diffusion pipeline loaded and mounted to GPU!")
except Exception as e:
    print(f"Falling back to Wan2.1: {e}")
    from diffusers import WanPipeline
    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
        torch_dtype=torch.float16
    )
    pipe.to("cuda")
    print("✅ Wan 2.1 pipeline loaded!")

In [ ]:
# 4. Batch Inference Across All 10 Shots
from PIL import Image

output_dir = Path("/kaggle/working/outputs/shots")
output_dir.mkdir(parents=True, exist_ok=True)

shots = storyboard["shots"]
telemetry = []

for shot in shots:
    shot_id = shot["shot_id"]
    title = shot["title"]
    motion_prompt = shot["motion_prompt"]
    duration_sec = shot["duration_seconds"]
    num_frames = int(duration_sec * 24)  # 24 fps
    
    keyframe_path = repo_dir / f"outputs/keyframes/{shot_id}.png"
    output_clip_path = output_dir / f"{shot_id}.mp4"
    
    print(f"\nProcessing {shot_id}: {title}")
    print(f"  Motion: {motion_prompt}")
    print(f"  Target Duration: {duration_sec}s ({num_frames} frames @ 24fps)")
    
    if not keyframe_path.exists():
        print(f"  [WARN] Keyframe {keyframe_path} not found. Skipping.")
        continue
        
    init_image = Image.open(keyframe_path).convert("RGB").resize((832, 480))
    
    t0 = time.time()
    try:
        # Generate video
        if hasattr(pipe, "unet"):
            # Stable Video Diffusion pipeline
            output_frames = pipe(init_image, decode_chunk_size=8, generator=torch.Generator("cuda").manual_seed(42)).frames[0]
        else:
            # Wan pipeline
            output_frames = pipe(prompt=motion_prompt, num_frames=min(num_frames, 81), num_inference_steps=25).frames[0]
        
        # Export clip to MP4
        from diffusers.utils import export_to_video
        export_to_video(output_frames, str(output_clip_path), fps=24)
        
        latency = round(time.time() - t0, 2)
        print(f"  [✓] {output_clip_path.name} saved in {latency}s")
        telemetry.append({"shot_id": shot_id, "latency": latency, "status": "success"})
    except Exception as err:
        print(f"  [X] Failed: {err}")
        telemetry.append({"shot_id": shot_id, "error": str(err), "status": "failed"})

print("\n--- Batch Run Completed ---")
print(json.dumps(telemetry, indent=2))

In [ ]:
# 5. Package Video Clips for Download Back to Local Workspace
!zip -j /kaggle/working/shots_bundle.zip /kaggle/working/outputs/shots/*.mp4
print("Created /kaggle/working/shots_bundle.zip!")
print("Download this file and unzip into your local ethiopian-heritage-short/outputs/shots/ directory.")